# SmartCare Option C - Preprocessing, Feature Engineering and Model Development

This notebook continues the verified dataset-understanding stage. It compares clinical-only and context-augmented feature sets while keeping the final 20% test set untouched until model selection and tuning are complete.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
    balanced_accuracy_score, classification_report, cohen_kappa_score,
    confusion_matrix, f1_score, mean_absolute_error, precision_score,
    recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
CLASS_NAMES = ["Low", "Medium", "High"]
CLASS_TO_INT = {name: index for index, name in enumerate(CLASS_NAMES)}
INT_TO_CLASS = {index: name for name, index in CLASS_TO_INT.items()}

## 1. Load data and establish the prediction target

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "smartcare_ai_dataset_1000.csv"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
TARGET = "disease_risk_level"

assert set(df[TARGET].unique()) == set(CLASS_NAMES)
assert df[TARGET].isna().sum() == 0

y = df[TARGET].map(CLASS_TO_INT).astype(int)
print(f"Loaded {len(df):,} records and {df.shape[1]} columns")
display(pd.DataFrame({"count": df[TARGET].value_counts().reindex(CLASS_NAMES),
                      "percentage": (df[TARGET].value_counts(normalize=True).reindex(CLASS_NAMES) * 100).round(1)}))

## 3. Create one untouched stratified test set

In [ ]:
train_index, test_index = train_test_split(
    np.arange(len(model_df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train_all = model_df.iloc[train_index].reset_index(drop=True)
X_test_all = model_df.iloc[test_index].reset_index(drop=True)
y_train = y.iloc[train_index].reset_index(drop=True)
y_test = y.iloc[test_index].reset_index(drop=True)

split_check = pd.DataFrame({
    "Full dataset": y.value_counts(normalize=True).sort_index(),
    "Training set": y_train.value_counts(normalize=True).sort_index(),
    "Test set": y_test.value_counts(normalize=True).sort_index(),
}, index=range(3))
split_check.index = CLASS_NAMES
display((split_check * 100).round(1).rename_axis("Risk class (%)"))
print(f"Training rows: {len(y_train)} | Untouched test rows: {len(y_test)}")

## 4. Reusable preprocessing pipelines

Numerical values are median-imputed and standardized. Categorical values are mode-imputed and one-hot encoded. All preprocessing is fitted inside each cross-validation fold.

In [ ]:
def build_preprocessor(numeric_features, categorical_features):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ], remainder="drop")

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=5, class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=350, min_samples_leaf=2, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        objective="multi:softprob", num_class=3, eval_metric="mlogloss",
        n_estimators=250, max_depth=3, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
        random_state=RANDOM_STATE, n_jobs=1
    ),
}

pipelines = {}
for feature_set_name, groups in feature_sets.items():
    preprocessor = build_preprocessor(groups["numeric"], groups["categorical"])
    for model_name, estimator in models.items():
        pipelines[(feature_set_name, model_name)] = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", clone(estimator)),
        ])
print(f"Prepared {len(pipelines)} leakage-safe model pipelines")

## 5. Compare four algorithms across both feature sets

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
    "roc_auc_ovr": "roc_auc_ovr",
}

comparison_rows = []
for (feature_set_name, model_name), pipeline in pipelines.items():
    selected_features = (feature_sets[feature_set_name]["numeric"]
                         + feature_sets[feature_set_name]["categorical"])
    cv_result = cross_validate(
        pipeline, X_train_all[selected_features], y_train,
        cv=cv, scoring=scoring, n_jobs=1, return_train_score=False,
    )
    row = {"Feature set": feature_set_name, "Model": model_name}
    for metric in scoring:
        values = cv_result[f"test_{metric}"]
        row[f"{metric} mean"] = values.mean()
        row[f"{metric} std"] = values.std(ddof=1)
    comparison_rows.append(row)
    print(f"Finished: {feature_set_name} | {model_name} | macro-F1={row['f1_macro mean']:.3f}")

comparison = pd.DataFrame(comparison_rows).sort_values("f1_macro mean", ascending=False)
display(comparison.round(3))